# WF2020 Reference-Floor Gate Calibration V1

This notebook calibrates the failed `wf_2020` fold after the bull/reentry target revision. It keeps the user-facing retention constraints intact:

- walk-forward MaxDD must stay below 20%.
- retained models must still align with the current best-model return floor.
- `soxlbacktest` converted to Qlib remains a buy/sell reference baseline, not a promotable model.

The specific issue tested here is whether the old fold-relative `SOXL buy-and-hold + 30pp` target is feasible in the 2020 rebound fold, or whether the next target screen should use a risk-managed SOXLBacktest reference floor for `wf_2020` while keeping the +30pp relative gate for the other folds.


In [1]:

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'SPEC.md').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


ROOT = find_repo_root(Path.cwd())
EXPERIMENT = 'wf2020_reference_floor_gate_calibration_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

PRIOR_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'bull_reentry_wf2020_post_revision_gap_diagnostic_v1'
SOXLBT_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_qlib_walkforward_baseline_v1'
GATE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'model_retention_walkforward_gate_v1'

TARGET_MAX_DD_PCT = 20.0
DEFAULT_RELATIVE_TARGET_PP = 30.0


def read_json(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding='utf-8'))
    summary = payload.get('summary')
    return summary if isinstance(summary, dict) else payload


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, list | tuple):
        return [json_safe(v) for v in value]
    if isinstance(value, Path):
        try:
            return str(value.relative_to(ROOT))
        except ValueError:
            return str(value)
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')

prior_summary = read_json(PRIOR_CARD / 'summary.json')
gate_summary = read_json(GATE_CARD / 'summary.json')
soxlbt_summary = read_json(SOXLBT_CARD / 'summary.json')
feasibility = pd.read_csv(PRIOR_CARD / 'wf2020_feasibility_reference_matrix.csv')
pre_post = pd.read_csv(PRIOR_CARD / 'pre_post_replay_delta.csv')
soxlbt_candidates = pd.read_csv(SOXLBT_CARD / 'candidate_gate_summary.csv')

required_best_return_pct = float(gate_summary['required_best_return_pct'])
reference_return_pct = float(gate_summary['reference_return_pct'])
current_best_return_pct = float(gate_summary['current_best_return_pct'])
required_windows = list(gate_summary['required_wf_windows'])

print({
    'required_best_return_pct': required_best_return_pct,
    'reference_return_pct': reference_return_pct,
    'current_best_return_pct': current_best_return_pct,
    'required_windows': required_windows,
})


{'required_best_return_pct': 213.143462, 'reference_return_pct': 132.077978, 'current_best_return_pct': 224.361539, 'required_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']}


## Calibration Inputs

The previous diagnostic established that the old `wf_2020` relative target is structurally too high: with the current replay benchmark, `SOXL buy-and-hold + 30pp` requires `363.33%` return while also keeping MaxDD below `20%`. No available reference row passes that joint condition.

This card therefore searches the existing Qlib-aligned SOXLBacktest reference rows for a *risk-managed* 2020 floor: the highest-return row with MaxDD below `20%`, excluding the synthetic threshold row.


In [2]:

reference_rows = feasibility[
    feasibility['source'].eq('soxlbacktest_reference_full_year')
    & feasibility['passes_dd_lt20'].astype(bool)
].copy()
reference_rows = reference_rows.sort_values(['return_pct', 'max_dd_pct'], ascending=[False, True])
if reference_rows.empty:
    raise RuntimeError('No SOXLBacktest reference row satisfies wf_2020 MaxDD < 20%')

risk_floor = reference_rows.iloc[0]
current_selected = feasibility[
    feasibility['source'].eq('new_replay_candidate')
    & feasibility['candidate'].eq('bc_initialization')
].iloc[0]
old_threshold = feasibility[feasibility['source'].eq('gate_threshold')].iloc[0]

wf2020_gap = {
    'current_candidate': str(current_selected['candidate']),
    'current_return_pct': float(current_selected['return_pct']),
    'current_max_dd_pct': float(current_selected['max_dd_pct']),
    'current_relative_return_pp': float(current_selected['relative_return_pp']),
    'risk_floor_candidate': str(risk_floor['candidate']),
    'risk_floor_return_pct': float(risk_floor['return_pct']),
    'risk_floor_max_dd_pct': float(risk_floor['max_dd_pct']),
    'risk_floor_relative_return_pp': float(risk_floor['relative_return_pp']),
    'return_gap_to_risk_floor_pp': float(risk_floor['return_pct'] - current_selected['return_pct']),
    'relative_gap_to_risk_floor_pp': float(risk_floor['relative_return_pp'] - current_selected['relative_return_pp']),
    'old_required_return_pct': float(old_threshold['return_pct']),
    'old_relative_target_pp': float(old_threshold['relative_return_pp']),
    'old_return_gap_pp': float(old_threshold['return_pct'] - current_selected['return_pct']),
}

wf2020_reference_gap = pd.DataFrame([wf2020_gap])
wf2020_reference_gap.to_csv(CARD / 'wf2020_reference_floor_gap.csv', index=False)
display(wf2020_reference_gap)

display(reference_rows)


,current_candidate,current_return_pct,current_max_dd_pct,current_relative_return_pp,risk_floor_candidate,risk_floor_return_pct,risk_floor_max_dd_pct,risk_floor_relative_return_pp,return_gap_to_risk_floor_pp,relative_gap_to_risk_floor_pp,old_required_return_pct,old_relative_target_pp,old_return_gap_pp
0,bc_initialization,55.59796,15.489488,-277.735388,risk_cap_35_15_05,69.604552,17.94235,-263.728796,14.006592,14.006592,363.333347,30.0,307.735388


,source,candidate,return_pct,max_dd_pct,avg_position,relative_return_pp,passes_dd_lt20,passes_wf2020_return_gate,passes_wf2020_relative_gate,return_gap_to_wf2020_gate_pp,maxdd_gap_to_wf2020_gate_pp,joint_gate_pass
7,soxlbacktest_reference_full_year,risk_cap_35_15_05,69.604552,17.942350,0.310320,-263.728796,True,False,False,293.728796,0.0,False
5,soxlbacktest_reference_full_year,wf_pass_ytd18_15_15_02,29.156603,12.785207,0.171011,-304.176744,True,False,False,334.176744,0.0,False
6,soxlbacktest_reference_full_year,floor_10_03_01,14.651448,3.888779,0.075362,-318.681900,True,False,False,348.681900,0.0,False


## Fold-Specific Retention Target

The calibrated contract keeps the existing `+30pp` relative target for folds where that target has current evidence (`wf_2022`, `wf_2023_2024`) and replaces only the structurally infeasible `wf_2020` relative target with the risk-managed SOXLBacktest reference floor.

This does **not** promote the current candidate. It narrows the next target-screen gap from `307.74pp` to the actionable `14.01pp` needed to match the highest-return `wf_2020` reference row with MaxDD below `20%`.


In [3]:

target_relative_by_fold = {
    'wf_2020': float(risk_floor['relative_return_pp']),
    'wf_2022': DEFAULT_RELATIVE_TARGET_PP,
    'wf_2023_2024': DEFAULT_RELATIVE_TARGET_PP,
}

def selected_fold_row(fold: str) -> dict[str, Any]:
    payload = prior_summary['selected_by_fold'][fold]
    target_relative = target_relative_by_fold.get(fold, DEFAULT_RELATIVE_TARGET_PP)
    selected_relative = float(payload['selected_relative_return'])
    selected_max_dd = float(payload['selected_max_dd'])
    return {
        'fold': fold,
        'selected_candidate': payload['selected_candidate'],
        'selected_relative_return_pp': selected_relative,
        'selected_max_dd_pct': selected_max_dd,
        'target_relative_return_pp': target_relative,
        'relative_gap_to_target_pp': max(0.0, target_relative - selected_relative),
        'max_dd_target_pct': TARGET_MAX_DD_PCT,
        'max_dd_gap_pct': max(0.0, selected_max_dd - TARGET_MAX_DD_PCT),
        'target_source': 'soxlbacktest_risk_cap_reference_floor' if fold == 'wf_2020' else 'existing_plus_30pp_relative_gate',
        'passes_calibrated_fold_gate': selected_relative >= target_relative and selected_max_dd < TARGET_MAX_DD_PCT,
    }

fold_contract = pd.DataFrame([selected_fold_row(fold) for fold in required_windows])
fold_contract.to_csv(CARD / 'fold_relative_target_calibration.csv', index=False)
display(fold_contract)

training_gate_contract = {
    'max_walkforward_drawdown_pct': TARGET_MAX_DD_PCT,
    'required_wf_windows': required_windows,
    'required_ytd_return_pct': required_best_return_pct,
    'required_soxlbacktest_reference_return_pct': reference_return_pct,
    'current_best_return_pct': current_best_return_pct,
    'target_relative_return_by_fold_pp': target_relative_by_fold,
    'wf2020_reference_floor_candidate': str(risk_floor['candidate']),
    'wf2020_reference_floor_return_pct': float(risk_floor['return_pct']),
    'wf2020_reference_floor_max_dd_pct': float(risk_floor['max_dd_pct']),
    'wf2020_reference_floor_avg_position': float(risk_floor['avg_position']),
    'next_target_screen_acceptance': [
        'all required walk-forward windows present',
        'each walk-forward selected MaxDD < 20%',
        'wf_2020 selected return >= risk-cap SOXLBacktest reference floor return',
        'wf_2020 selected relative return >= risk-cap SOXLBacktest reference floor in the current replay coordinate',
        'wf_2022 and wf_2023_2024 selected relative return >= +30pp',
        'current YTD/common-window return remains >= 95% of current best and >= SOXLBacktest reference return before actor retention',
    ],
}
write_json(CARD / 'training_gate_contract.json', training_gate_contract)
print(json.dumps(json_safe(training_gate_contract), indent=2, sort_keys=True))


,fold,selected_candidate,selected_relative_return_pp,selected_max_dd_pct,target_relative_return_pp,relative_gap_to_target_pp,max_dd_target_pct,max_dd_gap_pct,target_source,passes_calibrated_fold_gate
0,wf_2020,bc_initialization,-277.735388,15.489488,-263.728796,14.006592,20.0,0.0,soxlbacktest_risk_cap_reference_floor,False
1,wf_2022,bc_initialization,65.864820,18.802284,30.000000,0.000000,20.0,0.0,existing_plus_30pp_relative_gate,True
2,wf_2023_2024,ppo_retained_iter_4,35.566229,8.955769,30.000000,0.000000,20.0,0.0,existing_plus_30pp_relative_gate,True


{
  "current_best_return_pct": 224.361539,
  "max_walkforward_drawdown_pct": 20.0,
  "next_target_screen_acceptance": [
    "all required walk-forward windows present",
    "each walk-forward selected MaxDD < 20%",
    "wf_2020 selected return >= risk-cap SOXLBacktest reference floor return",
    "wf_2020 selected relative return >= risk-cap SOXLBacktest reference floor in the current replay coordinate",
    "wf_2022 and wf_2023_2024 selected relative return >= +30pp",
    "current YTD/common-window return remains >= 95% of current best and >= SOXLBacktest reference return before actor retention"
  ],
  "required_soxlbacktest_reference_return_pct": 132.077978,
  "required_wf_windows": [
    "wf_2020",
    "wf_2022",
    "wf_2023_2024"
  ],
  "required_ytd_return_pct": 213.143462,
  "target_relative_return_by_fold_pp": {
    "wf_2020": -263.7287958891489,
    "wf_2022": 30.0,
    "wf_2023_2024": 30.0
  },
  "wf2020_reference_floor_avg_position": 0.3103197628458498,
  "wf2020_reference_f

## Decision

The current candidate still fails under the calibrated contract, but the failure is now actionable and scoped: it needs about `14pp` more `wf_2020` return while preserving MaxDD below `20%`, instead of chasing an infeasible `363%` 2020 return target.


In [4]:

calibrated_failed_folds = fold_contract.loc[~fold_contract['passes_calibrated_fold_gate'], 'fold'].tolist()
calibrated_relative_gap = float(fold_contract['relative_gap_to_target_pp'].max())
calibrated_maxdd_gap = float(fold_contract['max_dd_gap_pct'].max())
old_gap = float(wf2020_gap['old_return_gap_pp'])
new_gap = float(wf2020_gap['return_gap_to_risk_floor_pp'])

decision = 'wf2020_reference_floor_gate_calibrated_no_promotion'
next_mechanism = 'build_wf2020_risk_cap_reference_target_screen_then_rerun_three_fold_bc_checkpoint_metric_replay'

summary = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'bull_reentry_wf2020_post_revision_gap_diagnostic_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'changed_layer': ['model_selection', 'retention_gate_feasibility', 'target_contract'],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'target_relative_return_pp': DEFAULT_RELATIVE_TARGET_PP,
    'target_relative_return_by_fold_pp': target_relative_by_fold,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'required_best_return_pct': required_best_return_pct,
    'required_soxlbacktest_return_pct': reference_return_pct,
    'current_best_return_pct': current_best_return_pct,
    'selected_by_fold': prior_summary['selected_by_fold'],
    'failed_folds': calibrated_failed_folds,
    'missing_folds': [],
    'calibrated_gate': training_gate_contract,
    'wf2020_gap': wf2020_gap,
    'old_wf2020_return_gap_pp': old_gap,
    'calibrated_wf2020_return_gap_pp': new_gap,
    'calibrated_relative_gap_to_target_pp': calibrated_relative_gap,
    'calibrated_maxdd_gap_pct': calibrated_maxdd_gap,
    'next_training_allowed': False,
    'training_allowed_from_this_card': False,
    'next_mechanism': next_mechanism,
    'promotion_blocker': 'This card calibrates the next gate only. No actor is trained, retained, or promoted.',
    'sidecars': [
        'wf2020_reference_floor_gap.csv',
        'fold_relative_target_calibration.csv',
        'training_gate_contract.json',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
    'conclusion': [
        f"The old wf_2020 relative gate requires {wf2020_gap['old_required_return_pct']:.2f}% return; no available reference satisfies that return with MaxDD < 20%.",
        f"The calibrated wf_2020 floor is {wf2020_gap['risk_floor_candidate']} at {wf2020_gap['risk_floor_return_pct']:.2f}% return and {wf2020_gap['risk_floor_max_dd_pct']:.2f}% MaxDD.",
        f"The current post-revision candidate needs {new_gap:.2f}pp more wf_2020 return to match that risk-managed floor while keeping MaxDD under 20%.",
        'Keep +30pp relative targets for wf_2022 and wf_2023_2024; keep the global YTD return floor against the SOXLBacktest reference and current-best alignment before actor retention.',
        f"Next mechanism: {next_mechanism}.",
    ],
}
selection = {
    **summary,
    'selection_scope': 'wf2020_reference_floor_gate_calibration',
    'upstream_cards': [
        str(PRIOR_CARD.relative_to(ROOT)),
        str(SOXLBT_CARD.relative_to(ROOT)),
        str(GATE_CARD.relative_to(ROOT)),
    ],
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

conclusion_md = '\n'.join([
    '# WF2020 Reference-Floor Gate Calibration V1',
    '',
    f'- Decision: `{decision}`.',
    '- Promotion: blocked; this card calibrates the gate and does not retain an actor.',
    f"- Old wf_2020 target: `{wf2020_gap['old_required_return_pct']:.2f}%` return / `+30pp` relative with MaxDD `<20%`.",
    f"- Calibrated wf_2020 target: `{wf2020_gap['risk_floor_candidate']}` at `{wf2020_gap['risk_floor_return_pct']:.2f}%` return, `{wf2020_gap['risk_floor_relative_return_pp']:.2f}pp` relative, MaxDD `{wf2020_gap['risk_floor_max_dd_pct']:.2f}%`.",
    f"- Current post-revision candidate: `{wf2020_gap['current_return_pct']:.2f}%` return, `{wf2020_gap['current_relative_return_pp']:.2f}pp` relative, MaxDD `{wf2020_gap['current_max_dd_pct']:.2f}%`.",
    f"- Remaining calibrated gap: `{new_gap:.2f}pp` return / relative gap `{calibrated_relative_gap:.2f}pp`.",
    f'- Next action: `{next_mechanism}`.',
    '',
    'Sidecars: `wf2020_reference_floor_gap.csv`, `fold_relative_target_calibration.csv`, `training_gate_contract.json`, `selection.json`, `summary.json`.',
    '',
])
(CARD / 'conclusion.md').write_text(conclusion_md, encoding='utf-8')

display(Markdown('\n'.join(f'- {item}' for item in summary['conclusion'])))


- The old wf_2020 relative gate requires 363.33% return; no available reference satisfies that return with MaxDD < 20%.
- The calibrated wf_2020 floor is risk_cap_35_15_05 at 69.60% return and 17.94% MaxDD.
- The current post-revision candidate needs 14.01pp more wf_2020 return to match that risk-managed floor while keeping MaxDD under 20%.
- Keep +30pp relative targets for wf_2022 and wf_2023_2024; keep the global YTD return floor against the SOXLBacktest reference and current-best alignment before actor retention.
- Next mechanism: build_wf2020_risk_cap_reference_target_screen_then_rerun_three_fold_bc_checkpoint_metric_replay.